# PDF Parsing Strategy

Financial documents come in very different shapes. A 10-K annual report is dense prose with scattered tables. An earnings slide deck is one slide per page with almost no prose. A quarterly press release is a short mix of both.

If we chunk everything the same way, we lose signal — slide text gets split mid-bullet, table rows get merged with prose paragraphs, and retrieval quality suffers.

This notebook implements a **classify-then-parse** pipeline:

```
PDF file
  └─► diagnose()    — extract structural metadata (pages, chars/page, table count)
        └─► classify()  — determine document type + chunking mode
              └─► parse_pdf()  — route to the right chunking strategy
                    ├─ per_page   → one Chunk per slide
                    └─ overlapping → windowed prose chunks + separate table chunks
```

All output is a flat `list[Chunk]` using the shared `Chunk` dataclass from `utils.py`, ready to be embedded and stored in ChromaDB.

## 1. Setup

In [ ]:
import pdfplumber
import re
from pathlib import Path

# utils.py lives in the same directory and provides the Chunk dataclass
# and make_chunk_id, which generates a deterministic ID from the source + position
from utils import Chunk, make_chunk_id

# ── Chunking constants ────────────────────────────────────────────────────────

# Maximum characters per prose chunk.
# ~800 chars is roughly 200 tokens — comfortably within most embedding model limits.
CHUNK_SIZE = 800

# How many characters from the end of one chunk to repeat at the start of the next.
# Overlap prevents a sentence that straddles a chunk boundary from being lost.
OVERLAP = 80

# Minimum characters for a chunk to be worth keeping.
# Filters out near-empty pages (headers, footers, whitespace-only extractions).
MIN_CHARS = 40

# How many characters to read from the first pages when classifying a document.
# 3 pages * ~3500 chars/page = enough to reliably find form-type keywords.
CLASSIFY_PAGES = 3

# Threshold: if chars_per_page is below this, treat the document as slide-format.
# Slides have sparse text — usually a title + a few bullets, rarely full paragraphs.
SLIDE_CHARS_THRESHOLD = 1500

print("Setup complete.")

## 2. Diagnose

Before we parse anything, we inspect the PDF to understand its structure. This gives us the raw numbers that the classifier uses to make decisions.

| Metric | Why it matters |
|---|---|
| `chars_per_page` | Low → slide deck; high → dense prose report |
| `tables` | High relative to pages → table-heavy document |
| `likely_scan` | True → text extraction won't work, need OCR |
| `size_kb` | Very large → likely has embedded images (e.g. NVIDIA annual report) |

In [ ]:
def diagnose(path: Path) -> dict:
    """
    Opens the PDF and collects structural statistics without reading all content.
    Returns a dict of metrics used by classify() to decide how to chunk.
    """
    total_chars = 0
    total_tables = 0
    page_count = 0

    with pdfplumber.open(path) as pdf:
        page_count = len(pdf.pages)

        for page in pdf.pages:
            # extract_text() returns None if pdfplumber finds no text objects on the page.
            # This happens on scanned pages where content is stored as images, not text.
            text = page.extract_text() or ""
            total_chars += len(text)

            # find_tables() uses pdfplumber's layout analysis to detect table structures.
            # It looks for ruling lines and aligned text columns.
            total_tables += len(page.find_tables())

    chars_per_page = total_chars / max(page_count, 1)

    # A document is likely a scan if almost no text was extracted.
    # Real text-based PDFs reliably have >100 chars/page even on sparse slides.
    # Scanned PDFs store pages as images — pdfplumber extracts nothing.
    likely_scan = chars_per_page < 100

    return {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "pages": page_count,
        "chars_per_page": round(chars_per_page, 1),
        "tables": total_tables,
        "likely_scan": likely_scan,
    }


# ── Run diagnose on every PDF in datasets/client ─────────────────────────────
PDF_DIR = Path("datasets/client")

diagnostics = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    diag = diagnose(pdf_path)
    diagnostics.append(diag)
    print(diag)

## 3. Classify

Classification uses two signals:

**Structural signal** (from `diagnose`):
- `chars_per_page < 1500` → slide deck regardless of content

**Text signal** (keyword scan of the first few pages):
- SEC form declarations like `"form 10-k"`, `"form 10-q"`, `"exhibit 99"` appear near the top of every regulated filing
- These are consistent across all companies because the SEC mandates specific cover-page language

The output is a `(document_type, chunking_mode)` tuple:
- `chunking_mode = "per_page"` → one chunk per slide
- `chunking_mode = "overlapping"` → windowed prose chunks + separate table chunks

In [ ]:
def classify(path: Path, diag: dict) -> tuple[str, str]:
    """
    Determines (document_type, chunking_mode) for a PDF.

    Reads only the first CLASSIFY_PAGES pages for the keyword scan,
    keeping this function fast even for large files like the NVIDIA annual report.
    """

    # ── Structural check first — no need to read text ─────────────────────────
    # If the average page has fewer than SLIDE_CHARS_THRESHOLD characters,
    # it's almost certainly a presentation-style document (slides, earnings decks).
    if diag["chars_per_page"] < SLIDE_CHARS_THRESHOLD:
        return "earnings_slides", "per_page"

    # ── Text keyword scan on the first few pages ──────────────────────────────
    # We lowercase the text so matching isn't case-sensitive.
    # SEC filings always declare their form type on the cover page.
    sample_text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages[:CLASSIFY_PAGES]:
            sample_text += (page.extract_text() or "").lower()

    # Check for SEC form-type declarations in order of specificity.
    # "form 10-k" always appears verbatim on the 10-K cover page.
    if "form 10-k" in sample_text or "annual report" in sample_text:
        return "10-K", "overlapping"

    if "form 10-q" in sample_text:
        return "10-Q", "overlapping"

    # Exhibit 99-1 is the standard EDGAR label for earnings press releases
    # filed as attachments to 8-K forms.
    if "exhibit 99" in sample_text:
        return "earnings_release", "overlapping"

    # ── Fallback: structural table-density heuristic ───────────────────────────
    # If tables outnumber pages, treat it as a financial report.
    if diag["tables"] / max(diag["pages"], 1) > 1.0:
        return "financial_report", "overlapping"

    return "unknown", "overlapping"


# ── Test classification on our PDFs ──────────────────────────────────────────
print(f"{'File':<45} {'Doc Type':<20} {'Mode'}")
print("-" * 80)
for diag in diagnostics:
    path = PDF_DIR / diag["file"]
    doc_type, mode = classify(path, diag)
    print(f"{diag['file']:<45} {doc_type:<20} {mode}")

## 4. Table Extraction

Tables are extracted as **separate chunks** from prose. This matters because:

- Financial tables have a very different structure to prose — a cell value like `"12,453"` is meaningless without its row and column headers
- Serialising to **markdown** preserves header context in every chunk, so the embedding captures `"Revenue | Q4 2024 | 12,453"` rather than just `"12,453"`
- Keeping them separate lets us filter at query time: a question like *"what was revenue?"* should prefer `content_type=table` chunks

pdfplumber's `find_tables()` detects table boundaries using ruling lines and column alignment, then `table_obj.extract()` returns a 2D list of cell strings.

In [ ]:
def table_to_chunk(table_data: list[list], page_num: int, source: str, idx: int, doc_type: str) -> Chunk | None:
    """
    Converts a pdfplumber table (a 2D list of cell values) into a Chunk.

    The table is serialized to markdown so that:
    - The header row is always part of the chunk text
    - Column context is preserved for every data row
    - The embedding model sees structured key-value pairs, not raw numbers
    """

    # Normalise cells: replace None (empty cell) with empty string, strip whitespace.
    # pdfplumber returns None for merged/empty cells.
    rows = [[str(cell or "").strip() for cell in row] for row in table_data]

    # Drop rows that are entirely empty — these are artifact rows from ruling lines.
    rows = [row for row in rows if any(cell for cell in row)]

    if not rows:
        return None  # nothing left after cleanup

    # Build markdown table.
    # The first row is treated as the header. If the table has no clear header
    # (e.g. pure data grids), it still works — the first row just becomes column labels.
    header = rows[0]
    body   = rows[1:]

    # Markdown table header row
    header_line = "| " + " | ".join(header) + " |"

    # Markdown separator row (required by the spec for valid markdown tables)
    separator   = "| " + " | ".join("---" for _ in header) + " |"

    # Data rows — pad short rows to match the header column count
    # so the table doesn't break when cells are merged or missing.
    data_lines = []
    for row in body:
        padded = row + [""] * (len(header) - len(row))
        data_lines.append("| " + " | ".join(padded[:len(header)]) + " |")

    md = "\n".join([header_line, separator] + data_lines)

    return Chunk(
        id=make_chunk_id(source, idx, md),
        text=md,
        source=source,
        page=page_num,
        content_type="table",
        document_type=doc_type,
    )


# ── Quick visual test ─────────────────────────────────────────────────────────
# Open the Google 10-Q (highest table density: 2.3 tables/page) and extract
# the first table to verify the markdown output looks correct.
test_path = PDF_DIR / "goog-10-q-q1-2025.pdf"

with pdfplumber.open(test_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        tables = page.find_tables()
        if tables:
            sample = table_to_chunk(tables[0].extract(), page_num, test_path.name, 0, "10-Q")
            if sample:
                print(f"Page {page_num} — first table as markdown:")
                print(sample.text[:600])
                break

## 5. Prose Chunking (Overlapping Windows)

Dense reports (10-K, 10-Q) are split into **overlapping windows** of text.

Why overlapping?
- A sentence that falls at the boundary between two chunks would be cut in half
- The OVERLAP window repeats the tail of chunk N at the start of chunk N+1
- This ensures every complete sentence is fully represented in at least one chunk

```
chunk 1: |=========================|----|
chunk 2:                      |----|=========================|
                               ^^^^  overlap region
```

Table regions are **masked out** before prose extraction — otherwise text inside table cells would appear in both the table chunk and the prose chunk.

In [ ]:
def extract_prose_text(page) -> str:
    """
    Extracts text from a pdfplumber page, excluding any areas covered by tables.

    pdfplumber works at the individual text-object level. Its filter() method
    accepts a predicate that receives each character/word object and returns
    True to keep it or False to discard it.

    We discard any object whose bounding box falls inside a detected table region.
    """
    # Get the bounding boxes of all tables on this page.
    # Each bbox is (x0, top, x1, bottom) in PDF coordinate space.
    table_bboxes = [t.bbox for t in page.find_tables()]

    if not table_bboxes:
        # No tables on this page — extract everything directly.
        return page.extract_text() or ""

    # Build a filtered view of the page that excludes table regions.
    # The lambda checks whether each text object's bounding box overlaps
    # with any of the table regions — if it does, the object is dropped.
    def not_in_table(obj):
        for (x0, top, x1, bottom) in table_bboxes:
            # An object is inside the table if its coordinates fall within the bbox.
            # We use a small tolerance (1pt) to avoid floating-point edge cases
            # where a text object sits exactly on the table border.
            if (x0 - 1 <= obj["x0"] and obj["x1"] <= x1 + 1 and
                top - 1 <= obj["top"] and obj["bottom"] <= bottom + 1):
                return False  # inside a table — exclude it
        return True  # outside all tables — keep it

    masked_page = page.filter(not_in_table)
    return masked_page.extract_text() or ""


def prose_to_chunks(text: str, page_num: int, source: str, start_idx: int, doc_type: str) -> list[Chunk]:
    """
    Splits a block of prose text into overlapping fixed-size chunks.

    The sliding window advances by (CHUNK_SIZE - OVERLAP) characters each step.
    This means the last OVERLAP characters of chunk N reappear at the start of
    chunk N+1, preventing sentences from being silently cut at chunk boundaries.
    """
    text = text.strip()

    # Skip near-empty text — footers, page numbers, whitespace artifacts.
    if len(text) < MIN_CHARS:
        return []

    chunks = []
    idx = start_idx
    pos = 0  # current position in the text string

    while pos < len(text):
        # Slice out a window of CHUNK_SIZE characters starting at pos.
        snippet = text[pos : pos + CHUNK_SIZE]

        if len(snippet) >= MIN_CHARS:
            chunks.append(Chunk(
                id=make_chunk_id(source, idx, snippet),
                text=snippet,
                source=source,
                page=page_num,
                content_type="prose",
                document_type=doc_type,
            ))
            idx += 1

        # Advance by CHUNK_SIZE minus OVERLAP so the next window overlaps
        # with the tail of the current one.
        pos += CHUNK_SIZE - OVERLAP

    return chunks


print("Prose chunking functions defined.")

## 6. Per-Page Chunking (Slides)

For slide-format documents (earnings presentations, investor decks), each page is one semantic unit. Merging text across pages would mix unrelated topics — slide 5 might be "Revenue" and slide 6 might be "Guidance", and concatenating them produces noise.

Strategy: **one Chunk per page**, containing all text extracted from that page. Tables on slide pages are small and tightly bound to their surrounding context, so we don't separate them — the whole page goes into a single chunk.

In [ ]:
def page_to_chunk(page, page_num: int, source: str, idx: int, doc_type: str) -> Chunk | None:
    """
    Extracts all text from a single slide page as one Chunk.

    Unlike the overlapping prose strategy, we do NOT mask table regions here.
    Slide tables are small (usually 3-6 cells) and provide essential context
    for the surrounding bullet points — keeping them together improves retrieval.
    """
    # extract_text() returns all text objects on the page in reading order.
    # For slides, this typically gives us the title + bullet points as one block.
    text = (page.extract_text() or "").strip()

    if len(text) < MIN_CHARS:
        # Skip near-empty slides — title-only pages, divider slides, etc.
        return None

    return Chunk(
        id=make_chunk_id(source, idx, text),
        text=text,
        source=source,
        page=page_num,
        content_type="prose",
        document_type=doc_type,
    )


print("Per-page chunking function defined.")

## 7. Main Parser

The `parse_pdf` function is the entry point. It:
1. Runs `diagnose` to get structural metadata
2. Runs `classify` to determine document type and chunking mode
3. Routes each page through the correct chunking strategy
4. Returns a flat `list[Chunk]`

The `idx` counter is shared across all pages so every chunk in a document gets a unique index, which is used by `make_chunk_id` to guarantee no ID collisions.

In [ ]:
def parse_pdf(path: Path) -> list[Chunk]:
    """
    Full pipeline: diagnose → classify → chunk.
    Returns a flat list of Chunks ready for embedding and storage.
    """

    source = path.name  # used as the source field in every Chunk

    # Step 1: Collect structural metadata.
    # This opens the file once to count pages, characters, and tables.
    diag = diagnose(path)

    # Step 2: Determine how to parse this document.
    # Returns (doc_type, chunking_mode) e.g. ("10-K", "overlapping")
    doc_type, mode = classify(path, diag)

    # Scanned PDFs have no extractable text — pdfplumber returns empty strings.
    # We flag them and skip rather than produce empty chunks.
    # (OCR handling would be added here in a production pipeline.)
    if diag["likely_scan"]:
        print(f"  [SKIP] {source} — appears to be a scanned document (OCR not implemented)")
        return []

    chunks = []
    idx = 0  # running counter for unique chunk IDs across all pages

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):

            if mode == "per_page":
                # ── Slide mode: one chunk per page ────────────────────────────
                # Each slide is self-contained, so we don't split or overlap.
                chunk = page_to_chunk(page, page_num, source, idx, doc_type)
                if chunk:
                    chunks.append(chunk)
                    idx += 1

            else:
                # ── Overlapping mode: tables first, then prose ─────────────────

                # Tables are extracted first so we know their bounding boxes.
                # We process tables before prose because the prose extraction
                # step uses those bboxes to mask out table regions.
                for table_obj in page.find_tables():
                    chunk = table_to_chunk(table_obj.extract(), page_num, source, idx, doc_type)
                    if chunk:
                        chunks.append(chunk)
                        idx += 1

                # Extract prose text from the page, with table regions masked out
                # to avoid double-counting text that also appears in a table chunk.
                prose_text = extract_prose_text(page)

                # Split the prose into overlapping windows and append to chunks.
                new_prose = prose_to_chunks(prose_text, page_num, source, idx, doc_type)
                chunks.extend(new_prose)
                idx += len(new_prose)

    return chunks


print("parse_pdf defined.")

## 7.5 HTM Parsing (SEC EDGAR Filings)

The `datasets/client` directory also contains SEC EDGAR HTML filings (`.htm`). These have the same financial content as PDFs but are delivered as HTML — tables are `<table>` elements and prose is running text between them.

The HTM pipeline reuses the same building blocks:

| Step | HTM | PDF equivalent |
|---|---|---|
| Classify | Read form type from filename | Keyword-scan first pages |
| Tables | Extract `<table>` elements → `table_to_chunk` | pdfplumber `find_tables()` → `table_to_chunk` |
| Prose | `get_text()` after removing tables → `prose_to_chunks` | `extract_prose_text()` → `prose_to_chunks` |

No `diagnose` step is needed — there are no pages to measure and the form type is already in the filename.

One EDGAR-specific concern: these filings use deeply **nested tables** for layout (indentation, two-column text). We skip any `<table>` that is a descendant of another `<table>` and only process top-level data tables.

In [ ]:
all_chunks: list[Chunk] = []

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = parse_pdf(pdf_path)
    all_chunks.extend(chunks)
    n_tables = sum(1 for c in chunks if c.content_type == "table")
    n_prose  = sum(1 for c in chunks if c.content_type == "prose")
    print(f"{pdf_path.name:<45}  {len(chunks):>4} chunks  ({n_tables} tables, {n_prose} prose)")

for htm_path in sorted(PDF_DIR.glob("*.htm")):
    chunks = parse_htm(htm_path)
    all_chunks.extend(chunks)
    n_tables = sum(1 for c in chunks if c.content_type == "table")
    n_prose  = sum(1 for c in chunks if c.content_type == "prose")
    print(f"{htm_path.name:<45}  {len(chunks):>4} chunks  ({n_tables} tables, {n_prose} prose)")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

In [ ]:
def parse_htm(path: Path) -> list[Chunk]:
    """
    Parse an SEC EDGAR HTM filing into chunks.

    Tables are extracted first (reusing table_to_chunk for consistent markdown
    serialisation), removed from the parse tree, then the remaining text is
    passed through prose_to_chunks with the same overlapping window logic used
    for PDF prose.
    """
    source = path.name
    doc_type, _ = classify_htm(path)

    # Extract ticker and filing date from the filename for metadata.
    # Filename pattern: TICKER_FORMTYPE_DATE.htm  e.g. AMZN_10-K_2025-02-07.htm
    parts = path.stem.split("_")
    extra = {}
    if len(parts) >= 1:
        extra["ticker"] = parts[0]
    if len(parts) >= 3:
        extra["filing_date"] = parts[2]

    html = path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    # Remove non-content tags before any extraction.
    for tag in soup(["script", "style", "head"]):
        tag.decompose()

    chunks = []
    idx = 0

    # ── Extract top-level tables ───────────────────────────────────────────────
    # EDGAR HTML uses deeply nested tables for layout. We only extract tables
    # that are NOT nested inside another table — nested ones are layout artifacts.
    # Collect the list up front before any decompose() calls alter the tree.
    top_tables = [t for t in soup.find_all("table") if not t.find_parent("table")]

    for table_elem in top_tables:
        rows = extract_htm_table(table_elem)
        chunk = table_to_chunk(rows, None, source, idx, doc_type)
        if chunk:
            chunk.extra.update(extra)
            chunks.append(chunk)
            idx += 1
        # Remove from the tree so its text isn't double-counted in prose below.
        table_elem.decompose()

    # ── Extract prose ──────────────────────────────────────────────────────────
    # After tables are removed, get_text() gives us only the narrative text.
    # We collapse blank lines that result from removed tags.
    raw_text = soup.get_text(separator="\n", strip=True)
    lines = [line for line in raw_text.splitlines() if line.strip()]
    prose_text = "\n".join(lines)

    prose_chunks = prose_to_chunks(prose_text, None, source, idx, doc_type)
    for c in prose_chunks:
        c.extra.update(extra)
    chunks.extend(prose_chunks)

    return chunks


print("parse_htm defined.")

## 8. Run the Pipeline

Parse every PDF in `datasets/client/` and collect all chunks.

In [ ]:
all_chunks: list[Chunk] = []

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    chunks = parse_pdf(pdf_path)
    all_chunks.extend(chunks)

    # Show a per-file summary: total chunks and how many are tables vs prose.
    n_tables = sum(1 for c in chunks if c.content_type == "table")
    n_prose  = sum(1 for c in chunks if c.content_type == "prose")
    print(f"{pdf_path.name:<45}  {len(chunks):>4} chunks  ({n_tables} tables, {n_prose} prose)")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

## 9. Inspect the Output

Spot-check the chunks to make sure the content and metadata look right before we move on to embedding.

In [ ]:
import pandas as pd

# Build a summary DataFrame — one row per chunk
rows = [
    {
        "source":        c.source,
        "document_type": c.document_type,
        "content_type":  c.content_type,
        "page":          c.page,
        "chars":         len(c.text),
    }
    for c in all_chunks
]

df = pd.DataFrame(rows)

print("=== Chunk counts by document type and content type ===")
print(df.groupby(["document_type", "content_type"]).size().unstack(fill_value=0))

print("\n=== Chunk size distribution (characters) ===")
print(df["chars"].describe().round(1))

In [ ]:
# ── Sample a prose chunk from each document type ──────────────────────────────
# This lets you verify the text looks clean and coherent before embedding.

for doc_type in df["document_type"].unique():
    # Find the first prose chunk from this document type
    sample = next(
        (c for c in all_chunks if c.document_type == doc_type and c.content_type == "prose"),
        None
    )
    if sample:
        print(f"\n{'─'*60}")
        print(f"Doc type : {doc_type}")
        print(f"Source   : {sample.source}  (page {sample.page})")
        print(f"Chunk ID : {sample.id}")
        print(f"Text     : {sample.text[:300]}...")

In [ ]:
# ── Sample a table chunk ───────────────────────────────────────────────────────
# Verify that markdown tables have headers and are readable.

table_chunks = [c for c in all_chunks if c.content_type == "table"]
if table_chunks:
    sample = table_chunks[0]
    print(f"Source : {sample.source}  (page {sample.page})")
    print(f"Type   : {sample.document_type}")
    print()
    # Show the full markdown table — should have a header row, separator, and data rows
    print(sample.text[:800])

## 10. Check for ID Collisions

Every chunk needs a unique ID before it can be stored in ChromaDB — duplicate IDs cause silent overwrites. We verify here that `make_chunk_id` produced no collisions across the entire corpus.

In [ ]:
# Collect all IDs and check for duplicates.
# A set has no duplicates, so if its length matches the list length, all IDs are unique.
all_ids = [c.id for c in all_chunks]
unique_ids = set(all_ids)

n_total     = len(all_ids)
n_unique    = len(unique_ids)
n_duplicate = n_total - n_unique

print(f"Total chunks : {n_total}")
print(f"Unique IDs   : {n_unique}")
print(f"Duplicates   : {n_duplicate}")

if n_duplicate == 0:
    print("\nAll IDs are unique — safe to ingest into ChromaDB.")
else:
    # Find which IDs are duplicated so we can debug the make_chunk_id inputs
    from collections import Counter
    dupes = [id_ for id_, count in Counter(all_ids).items() if count > 1]
    print(f"\nDuplicate IDs found: {dupes[:10]}")